In [366]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import ndcg_score
from sklearn.metrics import accuracy_score, precision_score, recall_score
from imodels import RuleFitRegressor
import optuna

# Load dataset
df = pd.read_csv('predrealh.csv')
print(df.groupby('Unnamed: 0').iou.max().mean())
df2 = pd.read_csv('predrealh_dbscan.csv')
print(df2.groupby('Unnamed: 0').iou.max().mean())
df = pd.concat([df,df2])

# Win logic
# df["is_top"] = df.groupby("Unnamed: 0")["iou"]\
#                    .transform(lambda x: x >= x.quantile(0.9))\
#                    .astype(int)
# sc =df[df['is_top']==1].groupby("Unnamed: 0").iou.mean().mean()
# print(sc)
# Prepare data
X = df[['Unnamed: 0','RealHC','PredHC','x','y','Stops','Uq']]
groups = df['Unnamed: 0'].values
X = X.drop(columns=['Unnamed: 0']).copy()
y = df['iou']
feature_cols = X.columns.tolist()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
y = y.values

0.9329114124356437
0.963152726293992


In [357]:
unique_groups = np.unique(groups)
train_groups, test_groups = train_test_split(unique_groups, test_size=0.2, random_state=42)

train_mask = np.isin(groups, train_groups)
test_mask = np.isin(groups, test_groups)

X_train, y_train = X_scaled[train_mask], y[train_mask]
X_test, y_test = X_scaled[test_mask], y[test_mask]
groups_train = groups[train_mask]

In [358]:
cv = GroupKFold(n_splits=3)
metrics = []
rulesets = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_train, y_train, groups_train), 1):
    print(f"\n📁 Fold {fold}")

    X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
    X_va, y_va = X_train[va_idx], y_train[va_idx]

    # Train model
    model = RandomForestRegressor(
        n_estimators=1000,
        max_depth=1,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_tr, y_tr)
    
    y_pred = model.predict(X_va)
    
    df_scores = df.iloc[va_idx][['Unnamed: 0','iou']]
    df_scores['pred'] = model.predict(X_va)
    sc = df_scores.sort_values(['Unnamed: 0','pred']).groupby(["Unnamed: 0"]).tail(1).iou.mean()
    print(sc)
    
    print(f"R2 Score:  {r2_score(y_va, y_pred):.3f}")
    print(f"RMSE:      {mean_squared_error(y_va, y_pred, squared=False):.3f}")


📁 Fold 1
0.87231276047111
R2 Score:  0.241
RMSE:      0.171

📁 Fold 2
0.8483704490177584
R2 Score:  -0.129
RMSE:      0.138

📁 Fold 3
0.7375907674023372
R2 Score:  -0.136
RMSE:      0.141


In [359]:
def objective(trial):
    # Hyperparameters to tune
    tree_size = trial.suggest_int("tree_size", 2, 8)
    max_rules = trial.suggest_int("max_rules", 10, 100)
    memory_par = trial.suggest_float("memory_par", 0.001, 0.1, log=True)

    sc_scores = []

    # GroupKFold cross-validation
    cv = GroupKFold(n_splits=3)
    for train_idx, val_idx in cv.split(X_train, y_train, groups_train):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        group_val = groups_train[val_idx]

        model = RuleFitRegressor(
            tree_size=tree_size,
            max_rules=max_rules,
            memory_par=memory_par,
            lin_standardise=True,
            exp_rand_tree_size=True,
            random_state=42,
        )
        model.fit(X_tr, y_tr)

        y_pred = model.predict(X_val)

        # Compute custom group-wise top prediction metric
        df_scores = pd.DataFrame({
            'group': group_val,
            'iou': y_val,
            'pred': y_pred
        })
        sc = df_scores.sort_values(['group', 'pred'])\
                      .groupby("group")\
                      .tail(1).iou.mean()
        sc_scores.append(sc)

    return np.mean(sc_scores)

In [360]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

[I 2025-06-14 13:16:35,645] A new study created in memory with name: no-name-c23d4ce0-f15b-4491-a6ff-71a304f7640c
[I 2025-06-14 13:17:52,939] Trial 0 finished with value: 0.7936209449420332 and parameters: {'tree_size': 8, 'max_rules': 34, 'memory_par': 0.0023616305967701706}. Best is trial 0 with value: 0.7936209449420332.
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.039e-03, tolerance: 5.237e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.352e-03, tolerance: 5.2

[I 2025-06-14 13:20:43,193] Trial 3 finished with value: 0.7702754247764476 and parameters: {'tree_size': 4, 'max_rules': 52, 'memory_par': 0.011095358792953326}. Best is trial 1 with value: 0.8142525390730854.
[I 2025-06-14 13:20:47,088] Trial 4 finished with value: 0.7964746633309981 and parameters: {'tree_size': 2, 'max_rules': 14, 'memory_par': 0.001160951453601181}. Best is trial 1 with value: 0.8142525390730854.
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.925e-03, tolerance: 5.237e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the sca

C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.956e-03, tolerance: 9.778e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.133e-02, tolerance: 8.905e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Du

C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.995e-01, tolerance: 5.010e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.991e-01, tolerance: 8.496e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Du

C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.187e-01, tolerance: 9.778e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.860e-01, tolerance: 5.010e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Du

...

C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.878e-01, tolerance: 5.010e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.706e-01, tolerance: 8.496e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Du

C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.769e-01, tolerance: 9.778e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.520e-01, tolerance: 5.010e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Du

C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.216e-02, tolerance: 8.997e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.331e-02, tolerance: 8.997e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Du

C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.809e-02, tolerance: 8.905e-03
  model = cd_fast.enet_coordinate_descent(
C:\Users\kamil\.conda\envs\aivcode\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:648: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.106e-02, tolerance: 8.905e-03
  model = cd_fast.enet_coordinate_descent(
[W 2025-06-14 14:00:54,368] Trial 28 failed with parameters: {'tree_size': 4, 'max_rules': 85, 'memory_par': 0.0017686672768865348} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\kamil\AppData\Roaming\Python\Python39\s

KeyboardInterrupt: 

Unnamed: 0
2     0.987102
6     0.958083
9     0.957617
13    0.947586
Name: iou, dtype: float64

In [361]:
def group_sizes(group_ids):
    _, counts = np.unique(group_ids, return_counts=True)
    return counts.tolist()

In [377]:
def objective(trial):
    params = {
        "objective": "regression",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 100),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 5.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 5.0),
        "random_state": 42,
    }

    cv = GroupKFold(n_splits=3)
    sc_scores = []

    for tr_idx, va_idx in cv.split(X_train, y_train, groups_train):
        X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
        X_va, y_va = X_train[va_idx], y_train[va_idx]
        g_va = groups_train[va_idx]

        booster = lgb.train(params, lgb.Dataset(X_tr, label=y_tr), num_boost_round=200)
        y_pred = booster.predict(X_va)

        df_val = pd.DataFrame({"group": g_va, "iou": y_va, "pred": y_pred})
        top1_sc = df_val.sort_values(['group', 'pred']).groupby("group").tail(1).iou.mean()
        sc_scores.append(top1_sc)

    return np.mean(sc_scores)

In [378]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("✅ Best Params:", study.best_params)

[I 2025-06-14 14:06:55,223] A new study created in memory with name: no-name-a47b571a-0ea7-41a2-9abf-9a8457d858e5
[I 2025-06-14 14:06:55,301] Trial 0 finished with value: 0.7429406785752821 and parameters: {'learning_rate': 0.18987151851586298, 'num_leaves': 72, 'min_data_in_leaf': 34, 'max_depth': 8, 'feature_fraction': 0.9979574289713485, 'lambda_l1': 4.862671472613657, 'lambda_l2': 2.9854084121434497}. Best is trial 0 with value: 0.7429406785752821.
[I 2025-06-14 14:06:55,439] Trial 1 finished with value: 0.7607808930079152 and parameters: {'learning_rate': 0.05624577914439373, 'num_leaves': 46, 'min_data_in_leaf': 40, 'max_depth': 7, 'feature_fraction': 0.8363274360965555, 'lambda_l1': 4.317200305473351, 'lambda_l2': 0.4065036515372855}. Best is trial 1 with value: 0.7607808930079152.
[I 2025-06-14 14:06:55,530] Trial 2 finished with value: 0.7768770055432629 and parameters: {'learning_rate': 0.1634130885416708, 'num_leaves': 118, 'min_data_in_leaf': 56, 'max_depth': 3, 'feature_fr

[I 2025-06-14 14:06:59,421] Trial 24 finished with value: 0.7690542790461937 and parameters: {'learning_rate': 0.0423783947938805, 'num_leaves': 17, 'min_data_in_leaf': 64, 'max_depth': 3, 'feature_fraction': 0.8763879825797524, 'lambda_l1': 1.1869167803354348, 'lambda_l2': 1.977665125950585}. Best is trial 22 with value: 0.7818332435775225.
[I 2025-06-14 14:06:59,560] Trial 25 finished with value: 0.7494925648371008 and parameters: {'learning_rate': 0.034867208920275256, 'num_leaves': 32, 'min_data_in_leaf': 52, 'max_depth': 3, 'feature_fraction': 0.8136061487772184, 'lambda_l1': 1.665839080466443, 'lambda_l2': 2.4762441341389305}. Best is trial 22 with value: 0.7818332435775225.
[I 2025-06-14 14:06:59,721] Trial 26 finished with value: 0.7678119948531418 and parameters: {'learning_rate': 0.06143332498234636, 'num_leaves': 44, 'min_data_in_leaf': 70, 'max_depth': 4, 'feature_fraction': 0.9091876958851329, 'lambda_l1': 1.0286060000890882, 'lambda_l2': 1.8759060278871156}. Best is trial

[I 2025-06-14 14:07:02,974] Trial 48 finished with value: 0.7474823905248984 and parameters: {'learning_rate': 0.1338816288310623, 'num_leaves': 98, 'min_data_in_leaf': 94, 'max_depth': 6, 'feature_fraction': 0.937014013520467, 'lambda_l1': 0.30408788793869584, 'lambda_l2': 3.7917267405435022}. Best is trial 30 with value: 0.7862432966985003.
[I 2025-06-14 14:07:03,159] Trial 49 finished with value: 0.7694936613114906 and parameters: {'learning_rate': 0.029728423075123033, 'num_leaves': 112, 'min_data_in_leaf': 81, 'max_depth': 4, 'feature_fraction': 0.6726134771106539, 'lambda_l1': 0.20873777460346118, 'lambda_l2': 2.6942618931149456}. Best is trial 30 with value: 0.7862432966985003.


✅ Best Params: {'learning_rate': 0.17332756262681437, 'num_leaves': 118, 'min_data_in_leaf': 37, 'max_depth': 3, 'feature_fraction': 0.6911228009255169, 'lambda_l1': 1.013000004523358, 'lambda_l2': 3.384427371268962}
